In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = "retail_lakehouse"

silver_orders = spark.table(
    f"{catalog}.{schema_name}.silver_orders"
)

products = spark.table(
    f"{catalog}.{schema_name}.bronze_products"
)

print("Silver orders:", silver_orders.count())
print("Products:", products.count())

display(silver_orders)
display(products)

In [0]:
#Join and calculate revenue
# Join orders with product details.
# Calculate revenue at the order-record level.

enriched_orders = (
    silver_orders
    .join(
        products,
        on="product_id",
        how="left"
    )
    .withColumn(
        "line_revenue",
        F.round(
            F.col("quantity") * F.col("unit_price"),
            2
        )
    )
)

display(enriched_orders)

In [0]:
# Aggregate revenue by date
completed_orders = enriched_orders.filter(
    F.col("status") == "Completed"
)

gold_daily_sales = (
    completed_orders
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("line_revenue"), 2).alias("total_revenue")
    )
    .orderBy("order_date")
)

display(gold_daily_sales)

In [0]:
# Aggregate by product
gold_product_performance = (
    completed_orders
    .groupBy(
        "product_id",
        "product_name",
        "category"
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("line_revenue"), 2).alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

display(gold_product_performance)

In [0]:
# Aggregate by region
gold_regional_sales = (
    completed_orders
    .groupBy("region")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("line_revenue"), 2).alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

display(gold_regional_sales)

In [0]:
# Save the Gold Delta tables
gold_tables = {
    "gold_daily_sales": gold_daily_sales,
    "gold_product_performance": gold_product_performance,
    "gold_regional_sales": gold_regional_sales
}

for table_name, dataframe in gold_tables.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            f"{catalog}.{schema_name}.{table_name}"
        )
    )

    print(f"Saved: {table_name}")

In [0]:
saved_daily = spark.table(
    f"{catalog}.{schema_name}.gold_daily_sales"
)

saved_products = spark.table(
    f"{catalog}.{schema_name}.gold_product_performance"
)

saved_regions = spark.table(
    f"{catalog}.{schema_name}.gold_regional_sales"
)

daily_revenue = (
    saved_daily
    .agg(F.sum("total_revenue"))
    .first()[0]
)

assert saved_daily.count() == 5
assert saved_products.count() == 4
assert saved_regions.count() == 4
assert round(float(daily_revenue), 2) == 4689.00

print("Gold validation passed.")
print("Daily sales rows:", saved_daily.count())
print("Product rows:", saved_products.count())
print("Regional rows:", saved_regions.count())
print("Total completed revenue:", daily_revenue)

In [0]:
# Business query
spark.sql(f"""
    SELECT
        product_name,
        category,
        total_orders,
        units_sold,
        total_revenue
    FROM `{catalog}`.`{schema_name}`.`gold_product_performance`
    ORDER BY total_revenue DESC
""").show(truncate=False)